# Finding QRS complex' metrics:

* Run `one_beat.py` on all 17 files before running this notebook.
* This notebook first has display of QRS metrics:
    - QRS onset
    - QRS' primary peak
    - QRS' secondary peak, if exists
    - QRS offset
    - baseline (median of the signal)
* If the algorithm fails to detect QRS onset and offset, it doesn't show that.
* The output of this function is a dataframe with one-subject-channel as a unit.
* More code in the notebook to get median QRS duration from all channels'; and pivot channel data
* Uses the metadata stored by `one_beat.py` to get the `exam_id` info
* Joins the original `exams.csv` to get the `age` and other data.
* Saves this file on the disk.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import sys
sys.path.append('../scripts')
from qrs_stats import find_qrs_points, get_all_qrs_metrics, trim_trace

%load_ext autoreload
%autoreload 2

In [ ]:
# loading one file to test the QRS metrics method
avg_beat_array = np.load("../data/one_beat_array_16.npy")
avg_beat_array.shape

## Plotting QRS points for checking

In [ ]:
# putting all together for plotting

def plot_qrs_points(
        avg_beat_array,
        subject,
        channel
):
    trace = avg_beat_array[subject, :, channel]
    signal = trim_trace(trace)
    prim_peak, sec_peak, qrs_onset_idx, qrs_offset_idx, baseline = find_qrs_points(signal)

    plt.figure(figsize=(12, 6))
    plt.plot(signal, alpha=0.75, label=f'Average Beat')
    if prim_peak:
        plt.scatter(prim_peak, signal[prim_peak], color='green', label='Primary Peak')
        plt.hlines(baseline, xmin=0, xmax=len(signal), color='orange', linestyle='--', label='Baseline')
    if qrs_onset_idx:
        plt.scatter(qrs_onset_idx, signal[qrs_onset_idx], color='red', marker='*', label='QRS Onset')
        plt.scatter(qrs_offset_idx, signal[qrs_offset_idx], color='purple', marker='*', label='QRS Offset')
    if sec_peak != prim_peak:
        plt.scatter(sec_peak, signal[sec_peak], color='blue', label='Secondary Peak')

    plt.title(f"Subject = {subject}, Channel = {channel}")
    plt.legend()
    plt.xlabel('Time')
    plt.ylabel('mV')
    plt.show()


In [ ]:
subject, channel = 11, 0
qrs_lengths = []
for channel in range(12):
    trace = avg_beat_array[subject, :, channel]
    signal = trim_trace(trace)
    pos_peak, neg_peak, qrs_onset_idx, qrs_offset_idx, baseline = find_qrs_points(signal)
    if qrs_onset_idx and qrs_offset_idx:
        qrs_lengths.append(qrs_offset_idx - qrs_onset_idx)
    else:
        qrs_lengths.append(None)
    plot_qrs_points(avg_beat_array, subject, channel)


In [ ]:
subject, channel = 11, 2
plot_qrs_points(avg_beat_array, subject, channel)

In [ ]:
subject, channel = 11, 2
trace = avg_beat_array[subject, :, channel]
signal = trim_trace(trace)
pos_peak, neg_peak, qrs_onset_idx, qrs_offset_idx, baseline = find_qrs_points(signal)
# print (pos_peak - neg_peak )

plot_qrs_points(avg_beat_array, subject, channel)

In [ ]:
subject, channel = 13, 0
plot_qrs_points(avg_beat_array, subject, channel)

In [ ]:
subject, channel = 123, 0
plot_qrs_points(avg_beat_array, subject, channel)

In [ ]:
subject, channel = 13, 0
plot_qrs_points(avg_beat_array, subject, channel)

In [ ]:
subject, channel = 3333, 1
plot_qrs_points(avg_beat_array, subject, channel)

In [ ]:
subject, channel = 3333, 7
plot_qrs_points(avg_beat_array, subject, channel)

In [ ]:
subject, channel = 13, 0
plot_qrs_points(avg_beat_array, subject, channel)

In [ ]:
subject, channel = 3333, 0
plot_qrs_points(avg_beat_array, subject, channel)

In [ ]:
subject, channel = 3333, 2
plot_qrs_points(avg_beat_array, subject, channel)


In [ ]:
subject, channel = 0, 0
plot_qrs_points(avg_beat_array, subject, channel)

In [ ]:
subject = 3333
plot_qrs_points(avg_beat_array, subject, 0)

In [ ]:
pixel = 10 / 4096
pixel * 1000 # in pixel in milli-seconds

## Putting it all together

In [ ]:
qrs_metrics, error_record = get_all_qrs_metrics()

In [ ]:
# qrs_metrics = pd.read_csv(metrics_beat_avg_file)
qrs_metrics.shape

In [ ]:
error_record.shape

In [ ]:
qrs_metrics.head()

In [ ]:
qrs_metrics.loc[:, 'qrs_dur'] = qrs_metrics['qrs_offset_idx'] - qrs_metrics['qrs_onset_idx']

In [ ]:
# qrs_amplitude is prim_peak_mV - sec_peak_mV if sec_peak_idx != prim_peak_idx, else prim_peak_mV - baseline

qrs_metrics.loc[:, 'qrs_amplitude'] = np.where(
    qrs_metrics['prim_peak_idx'] != qrs_metrics['sec_peak_idx'],
    abs(qrs_metrics['prim_peak_mV'] - qrs_metrics['sec_peak_mV']),
    abs(qrs_metrics['prim_peak_mV'] - qrs_metrics['baseline'])
)

qrs_metrics.shape

In [ ]:
qrs_metrics.head()

In [ ]:
qrs_metrics_subject = qrs_metrics.pivot_table(
    index=['file_num', 'subject_idx'],
    columns='channel',
    values='qrs_amplitude',
).reset_index()
qrs_metrics_subject.head()

In [ ]:
metadata = []
for n in range(1, 18):
    metadata_path = f"../data/average_beat_metadata_{n}.csv"
    df_meta = pd.read_csv(metadata_path)
    df_meta.loc[:, 'file_num'] = n
    metadata.append(df_meta.reset_index().rename(columns={'index': 'subject_idx'}))

metadata = pd.concat(metadata, ignore_index=True)

In [ ]:
# qrs_metrics_subject.drop(columns=['channel'], inplace=True)
channel_rename = {}
for col in range(12):
    channel_rename[col] = f"ampl_chan_{col}"
qrs_metrics_subject.rename(columns=channel_rename, inplace=True)
qrs_metrics_subject.head()

In [ ]:
qrs_duration = qrs_metrics.groupby(
    ['file_num', 'subject_idx']).agg(
        mean_dur = ('qrs_dur', np.mean),
        median_dur = ('qrs_dur', np.median),
        std_dur= ('qrs_dur', np.std),
        mean_ampl = ('qrs_amplitude', np.mean),
    ).reset_index()

qrs_metrics_subject = qrs_duration.merge(
    qrs_metrics_subject, on=['file_num', 'subject_idx']
)
qrs_metrics_subject.head()

In [ ]:
for col in channel_rename.values():
    na_filt = qrs_metrics_subject[col].isna()
    qrs_metrics_subject.loc[na_filt, col] = qrs_metrics_subject['mean_ampl']

# confirm all NAs are replaced
qrs_metrics_subject.info()

In [ ]:
metadata.head()

In [ ]:
qrs_metrics_subject = qrs_metrics_subject.merge(
    metadata[['file_num', 'subject_idx', 'exam_id', 'age', 'nn_predicted_age']],
    on=['file_num', 'subject_idx']
)

In [ ]:
df = pd.read_csv("../data/exams.csv")
df.head()

In [ ]:
qrs_metrics_subject = qrs_metrics_subject.merge(
    df[['exam_id', 'is_male', 'normal_ecg', 'timey']],
    on='exam_id'
)
qrs_metrics_subject.head()

In [ ]:
# saving this file to re-read in another notebook
qrs_metrics_subject.to_csv("../data/qrs_metrics_subject.csv", index=False)